# CV 4대 태스크 실습 — ③분할 실습1: Watershed 동전 분할

### 셀 1 — 패키지 설치 및 한글 폰트 설정

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy scikit-image scipy
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import abc
import os
from dataclasses import dataclass, field

import cv2
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import distance_transform_edt

nanum_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(nanum_path):
    fm.fontManager.addfont(nanum_path)
    plt.rcParams["font.family"] = fm.FontProperties(fname=nanum_path).get_name()
plt.rcParams["axes.unicode_minus"] = False

# ============================================================

### 셀 2 — ===== [도메인 계층 / Domain Layer] =====

In [ ]:
# 이 계층은 "분할(Segmentation)이란 무엇인가"만 정의한다.
# cv2/scipy 등 특정 구현 기술에는 전혀 의존하지 않는, 순수 타입/추상화 계층이다.
# (DIP의 토대: 상위 계층인 유스케이스는 이 추상 인터페이스에만 의존한다)
# ============================================================


@dataclass
class SegmentationResult:
    """분할 알고리즘 1회 실행의 결과를 표현하는 값 객체(Value Object).

    - label_map: 각 픽셀이 속한 세그먼트 번호(int)를 담은 2차원 배열.
      배경=1, 경계선=-1, 그 외 2 이상의 정수가 개별 물체(동전) 번호.
    - num_segments: 배경(1)을 제외하고 최종적으로 분리된 물체 개수.
    - boundary_overlay_bgr: 경계선을 원본 위에 빨간색으로 표시한 BGR 시각화 이미지.
      (알고리즘의 "부가 산출물"이라 결과 객체에 함께 담아 유스케이스/표현 계층에 전달)
    """

    label_map: np.ndarray
    num_segments: int
    boundary_overlay_bgr: np.ndarray


class ImageSegmenter(abc.ABC):
    """'이미지를 픽셀 단위로 분할한다'는 핵심 행위만 정의하는 작은 인터페이스(ISP).

    시각화(색칠)나 파일 저장 같은 책임은 포함하지 않는다 — 오직 입력 이미지를
    받아 SegmentationResult를 돌려주는 단일 책임만 가진다(SRP).
    어떤 구체 알고리즘(Watershed, GrabCut, 향후 딥러닝 분할 모델 등)을 쓰든
    이 인터페이스만 구현하면 유스케이스 계층의 수정 없이 교체 가능하다(OCP/LSP).
    """

    @abc.abstractmethod
    def segment(self, image: np.ndarray) -> SegmentationResult:
        """입력 이미지(그레이스케일 등)를 분할하여 SegmentationResult를 반환한다."""
        raise NotImplementedError


# ============================================================

### 셀 3 — ===== [유스케이스 계층 / Use Case Layer] =====

In [ ]:
# 오직 ImageSegmenter 추상화에만 의존하며(DIP), cv2/scipy를 직접 호출하지 않는다.
# "입력 이미지를 분할해서 결과와 개수를 돌려준다"는 애플리케이션 흐름만 책임진다(SRP).
# ============================================================


class SegmentationUseCase:
    """이미지 분할 유스케이스.

    생성자 주입(Constructor Injection)으로 ImageSegmenter 구현체를 받는다.
    어떤 구체 알고리즘이 주입되는지는 전혀 알지 못하므로, Watershed를 GrabCut이나
    다른 알고리즘으로 교체해도 이 클래스는 한 줄도 바뀌지 않는다(OCP).
    """

    def __init__(self, segmenter: ImageSegmenter):
        self._segmenter = segmenter

    def run(self, image: np.ndarray) -> SegmentationResult:
        """주입된 분할기로 이미지를 분할해 결과를 반환한다."""
        return self._segmenter.segment(image)


# ============================================================

### 셀 4 — ===== [인프라/어댑터 계층 / Infrastructure Layer] =====

In [ ]:
# cv2, scipy 등 구체적인 CV 라이브러리 호출은 전부 이 계층에만 존재한다.
# WatershedSegmenter: 분할 "계산"만 담당 (SRP)
# SegmentationVisualizer: 분할 결과를 "색칠해 보여주는" 표현만 담당 (SRP)
# 두 책임을 하나의 클래스로 합치지 않음으로써 각자 변경 이유가 분리된다.
# ============================================================


class WatershedSegmenter(ImageSegmenter):
    """Watershed 알고리즘으로 맞닿은 물체(예: 동전)를 픽셀 단위로 분리하는 어댑터.

    ImageSegmenter 인터페이스의 구체 구현체(Infrastructure) — Domain/UseCase는
    이 클래스의 존재를 모른 채 ImageSegmenter 타입으로만 다룬다(DIP).
    """

    def __init__(self, area_cutoff: int = 4000, morph_iterations: int = 2, dilate_iterations: int = 3):
        # area_cutoff: 동전 내부 구멍(엠보싱 숫자/반사광)으로 간주할 배경-조각 최대 면적
        self._area_cutoff = area_cutoff
        self._morph_iterations = morph_iterations
        self._dilate_iterations = dilate_iterations

    def segment(self, image: np.ndarray) -> SegmentationResult:
        img_gray = image

        # 이진화 + 노이즈 제거
        # Otsu 임계값으로 동전(전경)과 배경을 자동으로 분리한다.
        _, thresh = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        kernel = np.ones((3, 3), np.uint8)
        opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=self._morph_iterations)

        # Watershed를 위한 "확실한 배경/전경/모름" 영역 계산
        # - sure_bg: 전경을 팽창(dilate)시켜 확실한 배경 영역을 구한다
        # - sure_fg: 거리 변환(distance transform)으로 동전 중심부(확실한 전경)를 구한다
        # - unknown: 그 사이(동전 경계 등 애매한 영역) = Watershed가 풀어야 할 부분
        sure_bg = cv2.dilate(opening, kernel, iterations=self._dilate_iterations)
        dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
        _, sure_fg = cv2.threshold(dist_transform, 0.5 * dist_transform.max(), 255, 0)
        sure_fg = np.uint8(sure_fg)
        unknown = cv2.subtract(sure_bg, sure_fg)

        # 마커(marker) 생성 — 동전마다 서로 다른 번호(레이블)를 부여
        num_markers, markers = cv2.connectedComponents(sure_fg)
        markers = markers + 1          # 배경을 0이 아닌 1로 만든다 (Watershed 규약)
        markers[unknown == 255] = 0    # "모름" 영역은 0으로 표시 — Watershed가 채워나갈 부분
        print(f"동전 후보 개수(연결 요소): {num_markers - 1}")

        # Watershed 알고리즘 실행 — 픽셀 단위로 각 동전의 경계를 정밀하게 찾는다
        img_color = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2BGR)
        markers_ws = cv2.watershed(img_color, markers.copy())
        img_color[markers_ws == -1] = [0, 0, 255]  # 경계선을 빨간색으로 표시

        num_segments = int(markers_ws.max())
        print(f"최종 분할된 동전 개수: {num_segments - 1}")

        return SegmentationResult(
            label_map=markers_ws,
            num_segments=num_segments,
            boundary_overlay_bgr=img_color,
        )


class SegmentationVisualizer:
    """SegmentationResult를 사람이 보기 좋은 컬러 이미지로 바꾸는 표현(Presenter) 전담 클래스.

    분할 "계산"(WatershedSegmenter)과는 완전히 분리된 책임이다(SRP) —
    이 클래스를 바꾸거나 다른 시각화 방식으로 교체해도 분할 알고리즘/유스케이스는
    전혀 영향받지 않는다.

    시각화 보정: 동전 표면의 각인된 숫자·반사광 때문에 생기는 작은 내부 구멍을
    주변과 같은 동전 색으로 메운다 (분할 "개수/경계"에는 영향 없음, 오직 컬러
    마스크 표시만 보정하는 순수 시각화 로직 — WatershedSegmenter에서 계산한
    원본 결과는 그대로 둔 채, 여기서만 메워서 보여준다).
    """

    def __init__(self, area_cutoff: int = 4000, seed: int = 42):
        self._area_cutoff = area_cutoff
        self._seed = seed

    def _fill_embossed_holes(self, result: SegmentationResult) -> np.ndarray:
        """동전 내부의 작은 배경-라벨 구멍(엠보싱/반사광)을 가장 가까운 동전 라벨로 메운 라벨맵을 반환."""
        markers_ws = result.label_map
        bg_label_mask = (markers_ws == 1).astype(np.uint8)
        n_bg, comp = cv2.connectedComponents(bg_label_mask)
        hole_mask = np.zeros_like(bg_label_mask, dtype=bool)
        for i in range(1, n_bg):
            if (comp == i).sum() < self._area_cutoff:
                hole_mask |= (comp == i)

        label_img = markers_ws.copy()
        label_img[hole_mask] = 0
        label_img[markers_ws == -1] = 0  # 경계선 너머의 '진짜' 동전 라벨을 찾도록 탐색 대상에서 제외
        to_fill = (label_img == 0)
        _, inds = distance_transform_edt(to_fill, return_distances=True, return_indices=True)
        nearest_labels = label_img[tuple(inds)]
        final_labels = markers_ws.copy()
        fill_target = hole_mask | (markers_ws == -1)
        final_labels[fill_target] = nearest_labels[fill_target]
        return final_labels

    def colorize(self, result: SegmentationResult) -> np.ndarray:
        """분할 결과를 동전마다 다른 색으로 색칠한 컬러 마스크 이미지를 생성한다."""
        final_labels = self._fill_embossed_holes(result)
        markers_ws = result.label_map
        num_segments = result.num_segments

        colored = np.full_like(result.boundary_overlay_bgr, 235)  # 배경 = 연회색
        rng = np.random.default_rng(self._seed)
        colors = rng.integers(60, 255, size=(num_segments + 2, 3))
        for seg_id in range(2, num_segments + 1):  # 1=배경, -1=경계선 이므로 2부터 시작
            colored[final_labels == seg_id] = colors[seg_id]
        colored[markers_ws == -1] = [255, 255, 255]  # 원래 Watershed 경계선은 흰 선으로 다시 표시
        return colored



### 셀 5 — 샘플 이미지 준비 — 서로 맞닿은 동전 사진

In [ ]:
# (동전끼리 겹쳐 있어 "탐지(사각 박스)"만으로는 개수를 셀 수 없고, 픽셀 단위 "분할"이 필요한 대표 사례)
# ===== [구성 루트 / Composition Root — Colab 실행 코드] =====
# 여기서부터는 구체 어댑터를 생성하고 유스케이스에 주입한 뒤 실행 → 시각화 → 저장까지
# 수행하는 "조립" 코드만 둔다. 파일 I/O·matplotlib 호출이 허용되는 유일한 계층이다.
from skimage import data as skdata

coins = skdata.coins()  # 흑백 동전 사진
cv2.imwrite("/content/coins.png", coins)
img_gray = coins
print("이미지 크기:", img_gray.shape)


### 셀 6 — 의존성 조립(Composition) — 구체 어댑터 생성 → 유스케이스에 주입

In [ ]:
segmenter: ImageSegmenter = WatershedSegmenter()          # 인프라 계층의 구체 구현체
segmentation_usecase = SegmentationUseCase(segmenter)      # 유스케이스는 추상 타입만 알고 있음
visualizer = SegmentationVisualizer()                       # 표현 전담 클래스 (분할기와 분리)


### 셀 7 — 유스케이스 실행

In [ ]:
result = segmentation_usecase.run(img_gray)
colored = visualizer.colorize(result)
img_color = result.boundary_overlay_bgr
num_segments = result.num_segments


### 셀 8 — 결과 시각화 (원본 / 경계 오버레이 / 컬러 마스크)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_gray, cmap="gray")
axes[0].set_title("원본 이미지")
axes[1].imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
axes[1].set_title("Watershed 경계선 (빨간선)")
axes[2].imshow(colored)
axes[2].set_title(f"픽셀 단위 분할 결과 ({num_segments - 1}개 동전)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig("/content/watershed_result.png", dpi=120, bbox_inches="tight")
plt.show()
print("저장 완료: /content/watershed_result.png")
